## SRP500559 / GSE263566

**paper:** [https://doi.org/10.1111/acel.70219](https://onlinelibrary.wiley.com/doi/full/10.1111/acel.70219) - The Human Cardiac “Age-OME”: Age-Specific Changes in Myocardial Molecular Expression, 2025

**date, curator:** 2025-09-16, Sara Carsanaro

**notes**
* manually added age and sex using supplementary table 1

### annotation summary
run this after annotation is complete

In [18]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Left ventricular myocardium,UBERON:0006566,left ventricle myocardium,perfect match


In [19]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,16 years,HsapDv:0000110,16-year-old stage,perfect match
1,17 years,HsapDv:0000111,17-year-old stage,perfect match
2,18 years,HsapDv:0000112,18-year-old stage,perfect match
3,19 years,HsapDv:0000113,19-year-old stage,perfect match
5,21 years,HsapDv:0000115,21-year-old stage,perfect match
6,23 years,HsapDv:0000117,23-year-old stage,perfect match
8,54 years,HsapDv:0000148,54-year-old stage,perfect match
10,55 years,HsapDv:0000149,55-year-old stage,perfect match
11,56 years,HsapDv:0000150,56-year-old stage,perfect match
12,60 years,HsapDv:0000154,60-year-old stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP500559"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_72782/3311601719.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:11: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
16 samples dont have attributes, try to find them somewhere else
100%|██

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX24200569,SRP500559,Illumina NovaSeq 6000,SRS20972425,,,HsapDv:0000110,16-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194555,Left ventricular myocardium,16 years,,,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_3,SAMN40901816,16,year,,,,,,,,,16/09/2025,"Cryopreserved human myocardium was powderised as described in the proteomic MS methods and weighed to approximately 30mg. Powderised myocardium was lysed and homogenised in 500 µL TRIzol (Invitrogen) using steel balls in the TissueLyser LT (Qiagen) and kept on dry ice between cycles (3-5 x 1 minute). 1-bromo-3-chloropopane (50 µL) was added to each sample and left to incubate for 5 minutes at RT, then spun at 14,000 x g at 4°C for 15 minutes to induce phase separation. RNA-containing aqueous phase was transferred to a sterile tube and an equal volume of isopropanol was added, mixed by inverting, and left for 1 hour at RT. RNA was pelleted at 14,000 x g at 18°C for 15 minutes, supernatant discarded. RNA was then washed in 4°C 70% ethanol twice discarding supernatant after spinning at 14,000 x g at 4°C for 10 minutes and left to dry at RT. RNA was DNase treated for 30 minutes at 37°C and washed again. Pelleted RNA was reconstituted in 15 µL 4°C nuclease free diethyl pyrocarbonate (DEPC). Concentration and quality were tested on NanoDrop (Thermo Fisher Scientific) to a standard of 260nm/280nm = 1.8-2.0 and 260nm/230nm 1.7-2.2. RNA integrity was assessed using an RNA Nano Chip on an Bioanalyzer (Agilent). RNA-seq libraries were prepared with Illumina Stranded Total RNA prep Ligation with Ribo Zero Plus (100ng input and 11 PCR cycles) according to manufacturer's instructions. Quality checks were performed using the Qubit dsDNA High Sensitivity Assay Kit and the Revvity LabChip GX Touch. The final sequencing pool was quantified using the Qubit dsDNA High Sensitivity Assay Kit after pooling all libraries equimolar into a single library pool. Sizing was checked using the Agilent High Sensitivity D1000 Tapestation system. The RNA-seq libraries were sequenced using a paired-end 250bp kit on a S4 flow cell of the Illumina NovaSeq 6000 with a final run concentration of 58pM and 1% PhiX.",,,,Left ventricular myocardium,,,,TRANSCRIPTOMIC,cDNA,SRR28601924,public
1,SRX24200562,SRP500559,Illumina NovaSeq 6000,SRS20972418,,,HsapDv:0000111,17-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194548,Left ventricular myocardium,17 years,,,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_4,SAMN40901823,17,year,,,,,,,,,16/09/2025,"Cryopreserved human myocardium was powderised as described in the proteomic MS methods and weighed to approximately 30mg. Powderised myocardium was lysed and homogenised in 500 µL TRIzol (Invitrogen) using steel balls in the TissueLyser LT (Qiagen) and kept on dry ice between cycles (3-5 x 1 minute). 1-bromo-3-chloropopane (50 µL) was added to each sample and left to incubate for 5 minutes at RT, then spun at 14,000 x g at 4°C for 15 minutes to induce phase separation. RNA-containing aqueous phase was transferred to a sterile tube and an equal volume of isopropanol was added, mixed by inverting, and left for 1 hour at RT. RNA was pelleted at 14,000 x g at 18°C for 15 minutes, supernatant discarded. RNA was then washed in 4°C 70% ethanol twice discardi

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Left ventricular myocardium']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0006566'
library.loc[:,'anatName'] = 'left ventricle myocardium'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX24200569,SRP500559,Illumina NovaSeq 6000,SRS20972425,UBERON:0006566,left ventricle myocardium,HsapDv:0000110,16-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194555,Left ventricular myocardium,16 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_3,SAMN40901816,16,year,,,,,,,,,16/09/2025,"Cryopreserved human myocardium was powderised as described in the proteomic MS methods and weighed to approximately 30mg. Powderised myocardium was lysed and homogenised in 500 µL TRIzol (Invitrogen) using steel balls in the TissueLyser LT (Qiagen) and kept on dry ice between cycles (3-5 x 1 minute). 1-bromo-3-chloropopane (50 µL) was added to each sample and left to incubate for 5 minutes at RT, then spun at 14,000 x g at 4°C for 15 minutes to induce phase separation. RNA-containing aqueous phase was transferred to a sterile tube and an equal volume of isopropanol was added, mixed by inverting, and left for 1 hour at RT. RNA was pelleted at 14,000 x g at 18°C for 15 minutes, supernatant discarded. RNA was then washed in 4°C 70% ethanol twice discarding supernatant after spinning at 14,000 x g at 4°C for 10 minutes and left to dry at RT. RNA was DNase treated for 30 minutes at 37°C and washed again. Pelleted RNA was reconstituted in 15 µL 4°C nuclease free diethyl pyrocarbonate (DEPC). Concentration and quality were tested on NanoDrop (Thermo Fisher Scientific) to a standard of 260nm/280nm = 1.8-2.0 and 260nm/230nm 1.7-2.2. RNA integrity was assessed using an RNA Nano Chip on an Bioanalyzer (Agilent). RNA-seq libraries were prepared with Illumina Stranded Total RNA prep Ligation with Ribo Zero Plus (100ng input and 11 PCR cycles) according to manufacturer's instructions. Quality checks were performed using the Qubit dsDNA High Sensitivity Assay Kit and the Revvity LabChip GX Touch. The final sequencing pool was quantified using the Qubit dsDNA High Sensitivity Assay Kit after pooling all libraries equimolar into a single library pool. Sizing was checked using the Agilent High Sensitivity D1000 Tapestation system. The RNA-seq libraries were sequenced using a paired-end 250bp kit on a S4 flow cell of the Illumina NovaSeq 6000 with a final run concentration of 58pM and 1% PhiX.",,,,Left ventricular myocardium,,,,TRANSCRIPTOMIC,cDNA,SRR28601924,public
1,SRX24200562,SRP500559,Illumina NovaSeq 6000,SRS20972418,UBERON:0006566,left ventricle myocardium,HsapDv:0000111,17-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194548,Left ventricular myocardium,17 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_4,SAMN40901823,17,year,,,,,,,,,16/09/2025,"Cryopreserved human myocardium was powderised as described in the proteomic MS methods and weighed to approximately 30mg. Powderised myocardium was lysed and homogenised in 500 µL TRIzol (Invitrogen) using steel balls in the TissueLyser LT (Qiagen) and kept on dry ice between cycles (3-5 x 1 minute). 1-bromo-3-chloropopane (50 µL) was added to each sample and left to incubate for 5 minutes at RT, then spun at 14,000 x g at 4°C for 15 minutes to induce phase separation. RNA-containing aqueous phase was transferred to a sterile tube and an equal volume of isopropanol was added, mixed by inverting, and left for 1 hour at RT.

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['16 years' '17 years' '18 years' '19 years' '21 years' '23 years'
 '54 years' '55 years' '56 years' '60 years' '62 years' '65 years']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
my_protocol = 'Illumina Stranded Total RNA Prep with Ribo-Zero Plus'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'ribo-minus'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX24200569,SRP500559,Illumina NovaSeq 6000,SRS20972425,UBERON:0006566,left ventricle myocardium,HsapDv:0000110,16-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194555,Left ventricular myocardium,16 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_3,SAMN40901816,16,year,,,,,,,,,16/09/2025,"Cryopreserved human myocardium was powderised as described in the proteomic MS methods and weighed to approximately 30mg. Powderised myocardium was lysed and homogenised in 500 µL TRIzol (Invitrogen) using steel balls in the TissueLyser LT (Qiagen) and kept on dry ice between cycles (3-5 x 1 minute). 1-bromo-3-chloropopane (50 µL) was added to each sample and left to incubate for 5 minutes at RT, then spun at 14,000 x g at 4°C for 15 minutes to induce phase separation. RNA-containing aqueous phase was transferred to a sterile tube and an equal volume of isopropanol was added, mixed by inverting, and left for 1 hour at RT. RNA was pelleted at 14,000 x g at 18°C for 15 minutes, supernatant discarded. RNA was then washed in 4°C 70% ethanol twice discarding supernatant after spinning at 14,000 x g at 4°C for 10 minutes and left to dry at RT. RNA was DNase treated for 30 minutes at 37°C and washed again. Pelleted RNA was reconstituted in 15 µL 4°C nuclease free diethyl pyrocarbonate (DEPC). Concentration and quality were tested on NanoDrop (Thermo Fisher Scientific) to a standard of 260nm/280nm = 1.8-2.0 and 260nm/230nm 1.7-2.2. RNA integrity was assessed using an RNA Nano Chip on an Bioanalyzer (Agilent). RNA-seq libraries were prepared with Illumina Stranded Total RNA prep Ligation with Ribo Zero Plus (100ng input and 11 PCR cycles) according to manufacturer's instructions. Quality checks were performed using the Qubit dsDNA High Sensitivity Assay Kit and the Revvity LabChip GX Touch. The final sequencing pool was quantified using the Qubit dsDNA High Sensitivity Assay Kit after pooling all libraries equimolar into a single library pool. Sizing was checked using the Agilent High Sensitivity D1000 Tapestation system. The RNA-seq libraries were sequenced using a paired-end 250bp kit on a S4 flow cell of the Illumina NovaSeq 6000 with a final run concentration of 58pM and 1% PhiX.",,,,Left ventricular myocardium,,,,TRANSCRIPTOMIC,cDNA,SRR28601924,public
1,SRX24200562,SRP500559,Illumina NovaSeq 6000,SRS20972418,UBERON:0006566,left ventricle myocardium,HsapDv:0000111,17-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194548,Left ventricular myocardium,17 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_4,SAMN40901823,17,year,,,,,,,,,16/09/2025,"Cryopreserved human myocardium was powderised as described in the proteomic MS methods and weighed to approximately 30mg. Powderised myocardium was lysed and homogenised in 500 µL TRIzol (Invitrogen) using steel balls in the TissueLyser LT (Qiagen) and kept on dry ice between cycles (3-5 x 1 minute). 1-bromo-3-chloropopane (50 µL) was added to each sample and left to incubate for 5 minutes at RT, then spun at 14,000 x g at 4°C for 15 minutes to induce phase separation. RNA-containing aqueous phase was transferred to a sterile tube and an equal volume of isopropanol was added, mixed by inverting, and left for 1 hour at RT.

#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2025-09-16'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX24200569,SRP500559,Illumina NovaSeq 6000,SRS20972425,UBERON:0006566,left ventricle myocardium,HsapDv:0000110,16-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194555,Left ventricular myocardium,16 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_3,SAMN40901816,16,year,,,,,,,,SAC,2025-09-16,"Cryopreserved human myocardium was powderised as described in the proteomic MS methods and weighed to approximately 30mg. Powderised myocardium was lysed and homogenised in 500 µL TRIzol (Invitrogen) using steel balls in the TissueLyser LT (Qiagen) and kept on dry ice between cycles (3-5 x 1 minute). 1-bromo-3-chloropopane (50 µL) was added to each sample and left to incubate for 5 minutes at RT, then spun at 14,000 x g at 4°C for 15 minutes to induce phase separation. RNA-containing aqueous phase was transferred to a sterile tube and an equal volume of isopropanol was added, mixed by inverting, and left for 1 hour at RT. RNA was pelleted at 14,000 x g at 18°C for 15 minutes, supernatant discarded. RNA was then washed in 4°C 70% ethanol twice discarding supernatant after spinning at 14,000 x g at 4°C for 10 minutes and left to dry at RT. RNA was DNase treated for 30 minutes at 37°C and washed again. Pelleted RNA was reconstituted in 15 µL 4°C nuclease free diethyl pyrocarbonate (DEPC). Concentration and quality were tested on NanoDrop (Thermo Fisher Scientific) to a standard of 260nm/280nm = 1.8-2.0 and 260nm/230nm 1.7-2.2. RNA integrity was assessed using an RNA Nano Chip on an Bioanalyzer (Agilent). RNA-seq libraries were prepared with Illumina Stranded Total RNA prep Ligation with Ribo Zero Plus (100ng input and 11 PCR cycles) according to manufacturer's instructions. Quality checks were performed using the Qubit dsDNA High Sensitivity Assay Kit and the Revvity LabChip GX Touch. The final sequencing pool was quantified using the Qubit dsDNA High Sensitivity Assay Kit after pooling all libraries equimolar into a single library pool. Sizing was checked using the Agilent High Sensitivity D1000 Tapestation system. The RNA-seq libraries were sequenced using a paired-end 250bp kit on a S4 flow cell of the Illumina NovaSeq 6000 with a final run concentration of 58pM and 1% PhiX.",,,,Left ventricular myocardium,,,,TRANSCRIPTOMIC,cDNA,SRR28601924,public
1,SRX24200562,SRP500559,Illumina NovaSeq 6000,SRS20972418,UBERON:0006566,left ventricle myocardium,HsapDv:0000111,17-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194548,Left ventricular myocardium,17 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_4,SAMN40901823,17,year,,,,,,,,SAC,2025-09-16,"Cryopreserved human myocardium was powderised as described in the proteomic MS methods and weighed to approximately 30mg. Powderised myocardium was lysed and homogenised in 500 µL TRIzol (Invitrogen) using steel balls in the TissueLyser LT (Qiagen) and kept on dry ice between cycles (3-5 x 1 minute). 1-bromo-3-chloropopane (50 µL) was added to each sample and left to incubate for 5 minutes at RT, then spun at 14,000 x g at 4°C for 15 minutes to induce phase separation. RNA-containing aqueous phase was transferred to a sterile tube and an equal volume of isopropanol was added, mixed by inverting, and left for 1 hour 

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [11]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX24200569,SRP500559,Illumina NovaSeq 6000,SRS20972425,UBERON:0006566,left ventricle myocardium,HsapDv:0000110,16-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194555,Left ventricular myocardium,16 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_3,SAMN40901816,16,year,,,,,,,,SAC,2025-09-16
1,SRX24200562,SRP500559,Illumina NovaSeq 6000,SRS20972418,UBERON:0006566,left ventricle myocardium,HsapDv:0000111,17-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194548,Left ventricular myocardium,17 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_4,SAMN40901823,17,year,,,,,,,,SAC,2025-09-16
2,SRX24200568,SRP500559,Illumina NovaSeq 6000,SRS20972424,UBERON:0006566,left ventricle myocardium,HsapDv:0000112,18-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194554,Left ventricular myocardium,18 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_5,SAMN40901817,18,year,,,,,,,,SAC,2025-09-16
3,SRX24200566,SRP500559,Illumina NovaSeq 6000,SRS20972422,UBERON:0006566,left ventricle myocardium,HsapDv:0000113,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194552,Left ventricular myocardium,19 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_8,SAMN40901819,19,year,,,,,,,,SAC,2025-09-16
4,SRX24200565,SRP500559,Illumina NovaSeq 6000,SRS20972421,UBERON:0006566,left ventricle myocardium,HsapDv:0000113,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194551,Left ventricular myocardium,19 years,perfect match,not documented,perfect match,F,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_7,SAMN40901820,19,year,,,,,,,,SAC,2025-09-16
5,SRX24200564,SRP500559,Illumina NovaSeq 6000,SRS20972419,UBERON:0006566,left ventricle myocardium,HsapDv:0000115,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194550,Left ventricular myocardium,21 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_9,SAMN40901821,21,year,,,,,,,,SAC,2025-09-16
6,SRX24200567,SRP500559,Illumina NovaSeq 6000,SRS20972423,UBERON:0006566,left ventricle myocardium,HsapDv:0000117,23-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194553,Left ventricular myocardium,23 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_11,SAMN40901818,23,year,,,,,,,,SAC,2025-09-16
7,SRX24200563,SRP500559,Illumina NovaSeq 6000,SRS20972420,UBERON:0006566,left ventricle myocardium,HsapDv:0000117,23-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194549,Left ventricular myocardium,23 years,perfect match,not documented,perfect match,F,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,ribo-minus,,,young_10,SAMN40901822,23,year,,,,,,,,SAC,2025-09-16
8,SRX24200555,SRP500559,Illumina NovaSeq 6000,SRS20972411,UBERON:0006566,left ventricle myocardium,HsapDv:0000148,54-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8194557,Left ventricular myocardium,54 years,perfect match,not documented,perfect match,F,,,9606,Illumina 

### experiment annotations

In [12]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP500559,The Human Cardiac “Age-OME”: Age-specific changes in myocardial molecular expression,"Ageing is one of the most significant risk factors for heart disease, however, it is still not clear how the human heart changes with age. Taking advantage of a unique set of pre-mortem, cryopreserved, non-diseased human hearts, we performed omics analyses (transcriptomics, proteomics, metabolomics, and lipidomics), coupled with biologically-informed computational modelling in younger (≤25 years old) and older hearts (≥50 years old) to describe the molecular landscape of human cardiac ageing. In older hearts, we observed a downregulation of proteins involved in calcium signalling and contractile apparatus. Furthermore, we found a potential dysregulation of central carbon generation of fuel, glycolysis and fatty acids oxidation, along with an increase in long-chain fatty acids. This study presents and analyses the first molecular data set of normal human cardiac ageing, which has relevant implications for understanding the human cardiac ageing process and the development of age-related heart disease. Overall design: The aim of this study was to perform multi-omics analyses on cryopreserved pre-mortem human left ventricular myocardium to uncover the molecular influence of cardiac ageing.",SRA,,,,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,,PRJNA1098073,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [13]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

16

In [14]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
#experiment.loc[:,'projectTags'] = '' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP500559,The Human Cardiac “Age-OME”: Age-specific changes in myocardial molecular expression,"Ageing is one of the most significant risk factors for heart disease, however, it is still not clear how the human heart changes with age. Taking advantage of a unique set of pre-mortem, cryopreserved, non-diseased human hearts, we performed omics analyses (transcriptomics, proteomics, metabolomics, and lipidomics), coupled with biologically-informed computational modelling in younger (≤25 years old) and older hearts (≥50 years old) to describe the molecular landscape of human cardiac ageing. In older hearts, we observed a downregulation of proteins involved in calcium signalling and contractile apparatus. Furthermore, we found a potential dysregulation of central carbon generation of fuel, glycolysis and fatty acids oxidation, along with an increase in long-chain fatty acids. This study presents and analyses the first molecular data set of normal human cardiac ageing, which has relevant implications for understanding the human cardiac ageing process and the development of age-related heart disease. Overall design: The aim of this study was to perform multi-omics analyses on cryopreserved pre-mortem human left ventricular myocardium to uncover the molecular influence of cardiac ageing.",SRA,total,,16,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,,PRJNA1098073,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [15]:
experiment.loc[:,'GSE'] = 'GSE263566'
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://onlinelibrary.wiley.com/doi/full/10.1111/acel.70219'
experiment.loc[:,'DOI'] = 'https://doi.org/10.1111/acel.70219'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP500559,The Human Cardiac “Age-OME”: Age-specific changes in myocardial molecular expression,"Ageing is one of the most significant risk factors for heart disease, however, it is still not clear how the human heart changes with age. Taking advantage of a unique set of pre-mortem, cryopreserved, non-diseased human hearts, we performed omics analyses (transcriptomics, proteomics, metabolomics, and lipidomics), coupled with biologically-informed computational modelling in younger (≤25 years old) and older hearts (≥50 years old) to describe the molecular landscape of human cardiac ageing. In older hearts, we observed a downregulation of proteins involved in calcium signalling and contractile apparatus. Furthermore, we found a potential dysregulation of central carbon generation of fuel, glycolysis and fatty acids oxidation, along with an increase in long-chain fatty acids. This study presents and analyses the first molecular data set of normal human cardiac ageing, which has relevant implications for understanding the human cardiac ageing process and the development of age-related heart disease. Overall design: The aim of this study was to perform multi-omics analyses on cryopreserved pre-mortem human left ventricular myocardium to uncover the molecular influence of cardiac ageing.",SRA,total,,16,Illumina Stranded Total RNA Prep with Ribo-Zero Plus,full_length,GSE263566,PRJNA1098073,,https://onlinelibrary.wiley.com/doi/full/10.1111/acel.70219,https://doi.org/10.1111/acel.70219,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [16]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [17]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [20]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 3.97 KiB | 19.00 KiB/s, done.
From https://gitlab.sib.swiss/Bgee/expression-annotations
   761b13d..9a75a0f  develop    -> origin/develop
Updating 761b13d..9a75a0f
Fast-forward
 RNA_Seq/RNASeqExperiment.tsv           |  3 ++-
 RNA_Seq/RNASeqLibrary.tsv              | 48 +++++++++++++++++++++-------------
 RNA_Seq/RNASeqLibrary_not_included.tsv | 16 ++++++++++++
 3 files changed, 48 insertions(+), 19 deletions(-)
The columns in the library file match
The columns in the experiment file match


#### view files

In [21]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
49817,SRX9892364,SRP302083,Illumina HiSeq 2500,SRS8071719,UBERON:0001068,skin of back,UBERON:0000066,fully formed stage,,"skin, dorsal skin tissue",adult,perfect match,not documented,other,NA,NA,,992332,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Triplophysa rosa transcriptome of brain T03 (w...,SAMN17372387,,,,,,,see https://pubpeer.com/publications/911186984...,,,ANN,2025-09-16
49818,SRX9892363,SRP302083,Illumina HiSeq 2500,SRS8071718,UBERON:0001068,skin of back,UBERON:0000066,fully formed stage,,"skin, dorsal skin tissue",adult,perfect match,not documented,other,NA,NA,,992332,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Triplophysa rosa transcriptome of brain T01 (w...,SAMN17372248,,,,,,,see https://pubpeer.com/publications/911186984...,,,ANN,2025-09-16
49819,SRX24200569,SRP500559,Illumina NovaSeq 6000,SRS20972425,UBERON:0006566,left ventricle myocardium,HsapDv:0000110,16-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Left ventricular myocardium,16 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zer...,full_length,ribo-minus,,,young_3,SAMN40901816,16,year,,,,,,,,SAC,2025-09-16
49820,SRX24200562,SRP500559,Illumina NovaSeq 6000,SRS20972418,UBERON:0006566,left ventricle myocardium,HsapDv:0000111,17-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Left ventricular myocardium,17 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zer...,full_length,ribo-minus,,,young_4,SAMN40901823,17,year,,,,,,,,SAC,2025-09-16
49821,SRX24200568,SRP500559,Illumina NovaSeq 6000,SRS20972424,UBERON:0006566,left ventricle myocardium,HsapDv:0000112,18-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Left ventricular myocardium,18 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zer...,full_length,ribo-minus,,,young_5,SAMN40901817,18,year,,,,,,,,SAC,2025-09-16
49822,SRX24200566,SRP500559,Illumina NovaSeq 6000,SRS20972422,UBERON:0006566,left ventricle myocardium,HsapDv:0000113,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Left ventricular myocardium,19 years,perfect match,not documented,perfect match,M,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zer...,full_length,ribo-minus,,,young_8,SAMN40901819,19,year,,,,,,,,SAC,2025-09-16
49823,SRX24200565,SRP500559,Illumina NovaSeq 6000,SRS20972421,UBERON:0006566,left ventricle myocardium,HsapDv:0000113,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Left ventricular myocardium,19 years,perfect match,not documented,perfect match,F,,,9606,Illumina Stranded Total RNA Prep with Ribo-Zer...,full_length,ribo-minus,,,young_7,SAMN40901820,19,year,,,,,,,,SAC,2025-09-16


In [22]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
933,SRP431619,Illumina sequencing of total skin samples of E...,Total skin samples were utilized to compare in...,SRA,partial,Bgee 1K,8,Ribo-Zero,full_length,,PRJNA953648,40674398,https://pmc.ncbi.nlm.nih.gov/articles/PMC12270...,10.1371/journal.pone.0328623,,removed libraries that were exposed to citric ...
934,SRP302083,Triplophysa rosa genome sequencing and assembly,Triplophysa rosa whole genome sequencing and a...,SRA,partial,Bgee 1K,12,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA673076,22775430,https://onlinelibrary.wiley.com/doi/10.1111/me...,10.3109/19401736.2012.696628,,"contact with authors in PubPeer, see https://p..."
935,SRP500559,The Human Cardiac “Age-OME”: Age-specific chan...,Ageing is one of the most significant risk fac...,SRA,total,,16,Illumina Stranded Total RNA Prep with Ribo-Zer...,full_length,GSE263566,PRJNA1098073,,https://onlinelibrary.wiley.com/doi/full/10.11...,https://doi.org/10.1111/acel.70219,,


### add annotations to git

In [23]:
! git pull

Already up to date.


In [24]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [25]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [26]:
! git add $git_experiment_path $git_library_path

In [27]:
! git commit -m $commit_message_exp

[develop 21f692f] adding annotated bulk experiment SRP500559
 2 files changed, 19 insertions(+), 2 deletions(-)


In [28]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.83 KiB | 967.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git
   9a75a0f..21f692f  develop -> develop


### add annotation folder and script to git

In [29]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

nothing added to commit but untracked files present (use "git add" to track)


1. run first two cells (annotation summary)
2. export as html

In [30]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push